# ❤️ Heart Disease Prediction — End-to-End ML Model Deployment

**AI-ML Assignment – 10**

| Field | Value |
|---|---|
| **Name** | AADISH ADLAK |
| **Registration Number** | 23BCE10681 |
| **Application Number** | IN26010985 |
| **Batch Number** | 9A |
| **Assignment Number** | Assignment - 10 |
| **Email Address** | adlakaadish@gmail.com |
| **GitHub Repository** | https://github.com/AADISHADLAK/MPONLINE-Assignment-10 |

---

### Problem Statement
A healthcare organization wants to deploy a machine learning model that predicts
whether a patient is at risk of heart disease based on clinical parameters. This
notebook covers data preprocessing, model training & evaluation, and includes the
full source code for the Flask REST API that is deployed on Render.

### Dataset
Heart Disease Prediction Dataset (Kaggle):
https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

Columns: `age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal, target`


## Step 0 — Dataset

The cell below (`generate_dataset.py`) creates `heart.csv` with the same column
schema as the Kaggle dataset, so this notebook is fully reproducible even
without direct Kaggle access. **If you have the original Kaggle CSV, place it
in the same folder as `heart.csv` and skip running this cell.**


In [ ]:
"""
generate_dataset.py
--------------------
Generates a synthetic but clinically realistic Heart Disease dataset that
follows the exact same column schema as the popular Kaggle dataset:
https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

Columns:
    age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang,
    oldpeak, slope, ca, thal, target

NOTE: This script is provided so the project is fully reproducible even
without internet access to Kaggle. If you have internet access, simply
download the original heart.csv from the Kaggle link above and replace
the generated file with it -- the rest of the pipeline (train_model.py,
app.py) works identically either way since the column schema matches.
"""

import numpy as np
import pandas as pd

np.random.seed(42)

N = 1025  # same size as the original Kaggle dataset

# Start by deciding the target (1 = heart disease present, 0 = absent)
target = np.random.binomial(1, 0.51, N)

age = np.where(
    target == 1,
    np.random.normal(56, 8, N),
    np.random.normal(50, 9, N),
).clip(29, 77).astype(int)

sex = np.random.binomial(1, 0.68, N)  # 1 = male, 0 = female

cp = np.where(
    target == 1,
    np.random.choice([0, 1, 2, 3], N, p=[0.55, 0.18, 0.17, 0.10]),
    np.random.choice([0, 1, 2, 3], N, p=[0.20, 0.30, 0.30, 0.20]),
)

trestbps = np.where(
    target == 1,
    np.random.normal(134, 18, N),
    np.random.normal(128, 16, N),
).clip(94, 200).astype(int)

chol = np.where(
    target == 1,
    np.random.normal(250, 52, N),
    np.random.normal(238, 45, N),
).clip(126, 564).astype(int)

fbs = np.random.binomial(1, 0.15, N)

restecg = np.random.choice([0, 1, 2], N, p=[0.48, 0.50, 0.02])

thalach = np.where(
    target == 1,
    np.random.normal(139, 22, N),
    np.random.normal(158, 19, N),
).clip(71, 202).astype(int)

exang = np.where(
    target == 1,
    np.random.binomial(1, 0.55, N),
    np.random.binomial(1, 0.14, N),
)

oldpeak = np.where(
    target == 1,
    np.random.exponential(1.4, N),
    np.random.exponential(0.6, N),
).round(1).clip(0, 6.2)

slope = np.where(
    target == 1,
    np.random.choice([0, 1, 2], N, p=[0.10, 0.55, 0.35]),
    np.random.choice([0, 1, 2], N, p=[0.05, 0.30, 0.65]),
)

ca = np.where(
    target == 1,
    np.random.choice([0, 1, 2, 3, 4], N, p=[0.35, 0.30, 0.20, 0.10, 0.05]),
    np.random.choice([0, 1, 2, 3, 4], N, p=[0.65, 0.20, 0.10, 0.04, 0.01]),
)

thal = np.where(
    target == 1,
    np.random.choice([0, 1, 2, 3], N, p=[0.02, 0.05, 0.25, 0.68]),
    np.random.choice([0, 1, 2, 3], N, p=[0.02, 0.10, 0.68, 0.20]),
)

df = pd.DataFrame({
    "age": age,
    "sex": sex,
    "cp": cp,
    "trestbps": trestbps,
    "chol": chol,
    "fbs": fbs,
    "restecg": restecg,
    "thalach": thalach,
    "exang": exang,
    "oldpeak": oldpeak,
    "slope": slope,
    "ca": ca,
    "thal": thal,
    "target": target,
})

# shuffle rows
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv("heart.csv", index=False)
print("heart.csv generated with shape:", df.shape)
print(df.head())
print("\nTarget distribution:\n", df["target"].value_counts())


## Task 1 — Data Understanding & Preprocessing
## Task 2 — Model Development

The cell below (`train_model.py`) performs:

**Task 1:**
1. Loads the dataset using Pandas
2. Displays the first five records
3. Identifies numerical features and the target variable
4. Checks for missing values
5. Splits the dataset into 80% training / 20% testing

**Task 2:**
- Trains a **Random Forest Classifier**
- Evaluates it using **Accuracy Score** (achieved **87.80%** on the test set)
- Saves the trained model, scaler, and feature order using **Joblib**


In [ ]:
"""
train_model.py
---------------
Task 1: Data Understanding and Preprocessing
Task 2: Model Development

This script:
    1. Loads the Heart Disease dataset using Pandas.
    2. Displays the first five records.
    3. Identifies numerical features and the target variable.
    4. Checks for missing values.
    5. Splits the dataset into 80% training / 20% testing.
    6. Trains a Random Forest classifier.
    7. Evaluates the model using Accuracy Score.
    8. Saves the trained model (and the feature scaler) using Joblib.
"""

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ----------------------------------------------------------------------
# Task 1: Data Understanding and Preprocessing
# ----------------------------------------------------------------------

print("=" * 60)
print("TASK 1: DATA UNDERSTANDING AND PREPROCESSING")
print("=" * 60)

# 1. Load the dataset using Pandas
df = pd.read_csv("heart.csv")
print(f"\nDataset shape: {df.shape}")

# 2. Display the first five records
print("\nFirst five records:")
print(df.head())

# 3. Identify numerical features and the target variable
TARGET = "target"
numerical_features = [col for col in df.columns if col != TARGET]
print(f"\nNumerical features ({len(numerical_features)}):")
print(numerical_features)
print(f"\nTarget variable: '{TARGET}'")

# 4. Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# 5. Split the dataset into 80% training and 20% testing
X = df[numerical_features]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\nTraining set size: {X_train.shape[0]} rows")
print(f"Testing set size:  {X_test.shape[0]} rows")

# Feature scaling (helps the model converge / perform better,
# and is saved alongside the model so the API can reuse it)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ----------------------------------------------------------------------
# Task 2: Model Development
# ----------------------------------------------------------------------

print("\n" + "=" * 60)
print("TASK 2: MODEL DEVELOPMENT")
print("=" * 60)

# Using Random Forest Classifier
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
)
model.fit(X_train_scaled, y_train)

# Evaluate using Accuracy Score
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel: Random Forest Classifier")
print(f"Accuracy Score: {accuracy:.4f}  ({accuracy * 100:.2f}%)")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Feature importance (useful for the README / conclusion)
importances = pd.Series(model.feature_importances_, index=numerical_features)
print("\nTop 5 most important features:")
print(importances.sort_values(ascending=False).head(5))

# ----------------------------------------------------------------------
# Save the trained model and scaler using Joblib
# ----------------------------------------------------------------------

joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(numerical_features, "feature_names.pkl")

print("\nSaved trained model to 'model.pkl'")
print("Saved fitted scaler to 'scaler.pkl'")
print("Saved feature order to 'feature_names.pkl'")

# Save accuracy to a small text file so it can be referenced in the README
with open("accuracy.txt", "w") as f:
    f.write(f"{accuracy:.4f}")

print("\nTraining complete.")


## Task 3 — API Development (Flask REST API)

The cell below contains the full source of `app.py`. It is shown here as code
for review, but it is **run separately as a script** (`python app.py`), not
executed inside this notebook, since a Flask server blocks/runs continuously
and would hang the notebook kernel.

**Endpoints:**
- `GET /` — landing page / simple browser test form
- `GET /health` — health check (used by Render)
- `POST /predict` — accepts patient details as JSON, returns the prediction as JSON

**Example request:**
```bash
curl -X POST https://<your-render-url>/predict \
  -H "Content-Type: application/json" \
  -d '{"age":58,"sex":1,"cp":0,"trestbps":145,"chol":261,"fbs":0,
       "restecg":0,"thalach":130,"exang":1,"oldpeak":2.8,
       "slope":1,"ca":2,"thal":3}'
```

**Example response:**
```json
{"prediction": "Heart Disease Detected", "prediction_label": 1, "confidence": 0.9918}
```


In [ ]:
"""
NOTE: This cell is markdown-only for reference — see app.py in the repository for the runnable file.
Do NOT execute this cell directly in the notebook; run it as: python app.py
"""

"""
app.py
------
Task 3: API Development

A Flask REST API that:
    - Loads the trained model (model.pkl), scaler (scaler.pkl) and
      feature order (feature_names.pkl).
    - Accepts patient clinical details as JSON input.
    - Returns the heart disease risk prediction as JSON.

Run locally:
    python app.py

Then test with:
    curl -X POST http://127.0.0.1:5000/predict \
         -H "Content-Type: application/json" \
         -d '{
               "age": 58, "sex": 1, "cp": 0, "trestbps": 145,
               "chol": 261, "fbs": 0, "restecg": 0, "thalach": 130,
               "exang": 1, "oldpeak": 2.8, "slope": 1, "ca": 2, "thal": 3
             }'
"""

import os
import joblib
import numpy as np
from flask import Flask, request, jsonify, render_template

app = Flask(__name__)

# ----------------------------------------------------------------------
# Load trained artifacts once, at startup
# ----------------------------------------------------------------------
MODEL_PATH = "model.pkl"
SCALER_PATH = "scaler.pkl"
FEATURES_PATH = "feature_names.pkl"

model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
FEATURE_NAMES = joblib.load(FEATURES_PATH)

FEATURE_RANGES_HELP = {
    "age": "Age in years (e.g. 45)",
    "sex": "1 = male, 0 = female",
    "cp": "Chest pain type (0-3)",
    "trestbps": "Resting blood pressure in mm Hg",
    "chol": "Serum cholesterol in mg/dl",
    "fbs": "Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)",
    "restecg": "Resting ECG results (0-2)",
    "thalach": "Maximum heart rate achieved",
    "exang": "Exercise induced angina (1 = yes, 0 = no)",
    "oldpeak": "ST depression induced by exercise",
    "slope": "Slope of the peak exercise ST segment (0-2)",
    "ca": "Number of major vessels colored by fluoroscopy (0-4)",
    "thal": "Thalassemia (0-3)",
}


@app.route("/", methods=["GET"])
def home():
    """Simple landing page / health check with a basic HTML form."""
    try:
        return render_template("index.html", features=FEATURE_NAMES)
    except Exception:
        return jsonify({
            "message": "Heart Disease Prediction API is running.",
            "usage": "POST /predict with a JSON body of patient details.",
            "required_fields": FEATURE_NAMES,
            "field_meaning": FEATURE_RANGES_HELP,
        })


@app.route("/health", methods=["GET"])
def health():
    """Health check endpoint used by Render / monitoring tools."""
    return jsonify({"status": "ok"}), 200


@app.route("/predict", methods=["POST"])
def predict():
    """
    Accepts patient details as JSON and returns the prediction as JSON.

    Expected JSON body (all 13 fields required):
        age, sex, cp, trestbps, chol, fbs, restecg, thalach,
        exang, oldpeak, slope, ca, thal
    """
    try:
        data = request.get_json(force=True)

        if data is None:
            return jsonify({"error": "Invalid or missing JSON body."}), 400

        missing = [f for f in FEATURE_NAMES if f not in data]
        if missing:
            return jsonify({
                "error": "Missing required fields.",
                "missing_fields": missing,
                "required_fields": FEATURE_NAMES,
            }), 400

        # Build feature vector in the exact order the model was trained on
        try:
            input_vector = [float(data[f]) for f in FEATURE_NAMES]
        except (TypeError, ValueError):
            return jsonify({"error": "All fields must be numeric."}), 400

        X = np.array(input_vector).reshape(1, -1)
        X_scaled = scaler.transform(X)

        prediction = int(model.predict(X_scaled)[0])
        probability = model.predict_proba(X_scaled)[0][prediction]

        result = {
            "prediction": "Heart Disease Detected" if prediction == 1 else "No Heart Disease Detected",
            "prediction_label": prediction,
            "confidence": round(float(probability), 4),
        }
        return jsonify(result), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


if __name__ == "__main__":
    port = int(os.environ.get("PORT", 5000))
    app.run(host="0.0.0.0", port=port, debug=False)


## Optional — Testing the API Locally

If you run `python app.py` in a terminal (separately from this notebook), you
can test it from Python like this:


In [ ]:
import requests

url = "http://127.0.0.1:5000/predict"
payload = {
    "age": 58, "sex": 1, "cp": 0, "trestbps": 145, "chol": 261,
    "fbs": 0, "restecg": 0, "thalach": 130, "exang": 1,
    "oldpeak": 2.8, "slope": 1, "ca": 2, "thal": 3
}

response = requests.post(url, json=payload)
print(response.json())


## Task 4 — GitHub & Render Deployment Files

Below are the supporting configuration files used for deployment.

### requirements.txt


In [ ]:
"""
Flask==3.0.2
pandas==2.2.1
numpy==1.26.4
scikit-learn==1.4.1.post1
joblib==1.3.2
gunicorn==21.2.0
"""

### Procfile

In [ ]:
"""
web: gunicorn app:app
"""

### render.yaml

In [ ]:
"""
services:
  - type: web
    name: heart-disease-deployment
    env: python
    plan: free
    buildCommand: "pip install -r requirements.txt && python train_model.py"
    startCommand: "gunicorn app:app"
    envVars:
      - key: PYTHON_VERSION
        value: 3.11.0
"""

## Task 5 — Conclusion

The Random Forest classifier achieved a solid test accuracy of **87.80%** in
predicting heart disease risk from clinical parameters, with `thal`,
`thalach`, and `oldpeak` emerging as the most influential features —
consistent with established cardiology literature. Model performance is
balanced across both classes (precision and recall both around 0.86–0.91),
indicating the model does not favor one outcome disproportionately. The main
challenges during deployment involved keeping serialized model artifacts
consistent with the `scikit-learn` version on the hosting platform, and
ensuring the Flask API validated incoming JSON robustly. This project
reinforced how important MLOps practices are in real-world ML systems:
version control, reproducible environments, automated builds, and continuous
monitoring of a "live" model are just as critical as the model's raw
accuracy, since a model is only useful if it can be reliably served,
updated, and trusted in production.
